# 00 — Data Exploration & Method Visualisation

Loads a single `.pkl` data file and produces diagnostic figures:

1. **Event frequency map** — where do extreme events occur?
2. **GMT ↔ temperature relationships** — for the best-attributed event
3. **Correctly attributed event** — distributions, SLP analogue map, local Ridge weight map
4. **Type I error example** — same diagnostics for a spuriously attributed counterfactual event

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/homerdurand/Attribution_challenge_2026/blob/main/00_exploration.ipynb)

**Set the paths in the CONFIG cell, then Run All.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import sys, os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.stats import norm
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

SRC_PATH = '/content/drive/MyDrive/ellis-attribution/src'
sys.path.insert(0, SRC_PATH)
from data_utils import extract_local_slp

# ── Cartopy feature helpers ───────────────────────────────────────
PROJ    = ccrs.PlateCarree()
LAND    = cfeature.NaturalEarthFeature('physical','land',   '50m', facecolor='#f0ebe3')
OCEAN   = cfeature.NaturalEarthFeature('physical','ocean',  '50m', facecolor='#cfe2f3')
COAST   = cfeature.NaturalEarthFeature('physical','coastline','50m',
                                        edgecolor='#444', facecolor='none', linewidth=0.6)
BORDERS = cfeature.NaturalEarthFeature('cultural','admin_0_boundary_lines_land','50m',
                                        edgecolor='#999', facecolor='none', linewidth=0.4)

def add_map_features(ax):
    ax.add_feature(OCEAN,   zorder=0)
    ax.add_feature(LAND,    zorder=1)
    ax.add_feature(COAST,   zorder=2)
    ax.add_feature(BORDERS, zorder=2)

In [ ]:
# ================================================================
#  CONFIG  —  edit here
# ================================================================
PKL_PATH    = '/content/drive/MyDrive/ellis-attribution/data/extracted_tasmax_nmem10_start1980_p99.9.pkl'
FIGURES_DIR = '/content/drive/MyDrive/ellis-attribution/figures'

# Member index to use (0 = first member)
MEMBER_IDX  = 0

# Analogue parameters
N_ANALOGUES  = 50
N_YEARS_POOL = 50

# SLP box half-widths (degrees)
ANALOGUE_HALF_DEG = 25.0   # box used for KNN analogue matching  -> 50x50 deg
DYNADJ_HALF_DEG   = 40.0   # box used for local Ridge DynAdj     -> 80x80 deg

# Ridge / PCA parameters
N_PCA_LOCAL  = 10
RIDGE_ALPHAS = np.logspace(-3, 8, 20)

# PN time window
WINDOW_BEFORE = 72   # months
WINDOW_AFTER  = 12
# ================================================================

os.makedirs(FIGURES_DIR, exist_ok=True)

with open(PKL_PATH, 'rb') as fh:
    full_data = pickle.load(fh)

d = full_data[MEMBER_IDX]
print(f'Member  : {d["member"]}')
print(f'Events  : {d["f_tas"].shape[1]}')
print(f'Timesteps: {len(d["times"])}')
print(f'SLP grid : {d["f_slp"].shape}')


## 1. Event frequency map

Pixel-wise count of timesteps belonging to an extreme event (factual run),
overlaid with event barycentres pooled across all loaded members.

In [ ]:
all_lats = np.concatenate([m['location'][:, 0] for m in full_data])
all_lons = np.concatenate([m['location'][:, 1] for m in full_data])

freq_map = d['event_frequency_map']   # xr.DataArray (lat, lon)
lats = freq_map.lat.values
lons = freq_map.lon.values
vals = freq_map.values.astype(float)
vals[vals == 0] = np.nan

fig, ax = plt.subplots(figsize=(14, 6), subplot_kw={'projection': PROJ})
add_map_features(ax)

pcm = ax.pcolormesh(lons, lats, vals, cmap='YlOrRd', vmin=1,
                     transform=PROJ, zorder=3)
ax.scatter(all_lons, all_lats, s=12, c='#2c3e50', alpha=0.5,
           transform=PROJ, zorder=4, label='Event barycentre')

plt.colorbar(pcm, ax=ax, label='Event count (timesteps)', shrink=0.7, pad=0.02)
ax.set_global()
ax.set_title(f'Extreme event frequency — member {d["member"]}', fontweight='bold')
ax.legend(loc='lower left', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'freq_map.png'), dpi=200, bbox_inches='tight')
plt.show()

## Auto-select events of interest

Scores every event with a quick thermodynamic PN (Gaussian) and selects:
- **Factual** event with the **highest PN** → correctly attributed
- **Counterfactual** event with the **highest PN** → worst Type I error

In [ ]:
def pn_gaussian(tas_win, cf_win, threshold):
    mu_f,  s_f  = norm.fit(tas_win)
    mu_cf, s_cf = norm.fit(cf_win)
    p_f  = norm.sf(threshold, mu_f,  s_f)
    p_cf = norm.sf(threshold, mu_cf, s_cf)
    return max(0.0, 1.0 - p_cf / p_f) if p_f > 0 else 0.0


def thermo_cf(tas, gmt):
    gmt_c = np.full_like(gmt, gmt[0])
    reg   = LinearRegression().fit(gmt[:, None], tas)
    return tas - reg.predict(gmt[:, None]) + reg.predict(gmt_c[:, None])


n_ev  = d['f_tas'].shape[1]
gmt   = d['gmt4_f']
pn_f  = np.zeros(n_ev)
pn_cf = np.zeros(n_ev)

for e in range(n_ev):
    for is_f in (True, False):
        idx_arr = d['idx_f'] if is_f else d['idx_c']
        tas_arr = d['f_tas'] if is_f else d['c_tas']
        val_arr = d['f_tas_vals'] if is_f else d['c_tas_vals']
        t_obs   = idx_arr[e]
        t0, t1  = max(0, t_obs - WINDOW_BEFORE), min(n_ev, t_obs + WINDOW_AFTER)
        tas_e   = tas_arr[:, e]
        cf      = thermo_cf(tas_e, gmt)
        pn      = pn_gaussian(tas_e[t0:t1], cf[t0:t1], val_arr[e])
        if is_f:
            pn_f[e]  = pn
        else:
            pn_cf[e] = pn

e_fact = int(np.argmax(pn_f))
e_type1 = int(np.argmax(pn_cf))

print(f'Best factual attribution : event {e_fact:3d}   PN={pn_f[e_fact]:.3f}')
print(f'Worst Type I error       : event {e_type1:3d}   PN={pn_cf[e_type1]:.3f}')

## 2. GMT ↔ temperature relationships

For the best-attributed factual event: time series of temperature and GMT (top),
and scatter with regression slope and correlation (bottom).

In [ ]:
e_idx   = e_fact
t_obs_f = d['idx_f'][e_idx]
tas_e   = d['f_tas'][:, e_idx]
ev_lat, ev_lon = d['location'][e_idx]
times_dt = pd.to_datetime(d['times'])

reg = LinearRegression().fit(gmt[:, None], tas_e)
r   = np.corrcoef(gmt, tas_e)[0, 1]
gmt_line = np.linspace(gmt.min(), gmt.max(), 200)

fig = plt.figure(figsize=(14, 6))
gs  = gridspec.GridSpec(1, 3, width_ratios=[2, 0.05, 1], wspace=0.3)

# ── Time series ───────────────────────────────────────────────────
ax_ts  = fig.add_subplot(gs[0])
ax_ts2 = ax_ts.twinx()

ax_ts.plot(times_dt, tas_e, color='#c0392b', lw=1.0, alpha=0.7, label='Tas')
ax_ts2.plot(times_dt, gmt,  color='#2980b9', lw=1.4, label='GMT (4yr)')
ax_ts.axvline(times_dt[t_obs_f], color='black', lw=1.5, ls='--', label='Event')

ax_ts.set_ylabel('Tas anomaly (K)', color='#c0392b')
ax_ts2.set_ylabel('GMT anomaly (K)', color='#2980b9')
ax_ts.set_xlabel('Year')
ax_ts.set_title(f'Event {e_idx}  |  lat={ev_lat:.1f}°  lon={ev_lon:.1f}°',
                fontweight='bold')
lines1, lab1 = ax_ts.get_legend_handles_labels()
lines2, lab2 = ax_ts2.get_legend_handles_labels()
ax_ts.legend(lines1 + lines2, lab1 + lab2, fontsize=8, loc='upper left')
ax_ts.grid(True, alpha=0.2)

# ── Scatter ───────────────────────────────────────────────────────
ax_sc = fig.add_subplot(gs[2])
ax_sc.scatter(gmt, tas_e, s=6, alpha=0.35, color='#7f8c8d')
ax_sc.plot(gmt_line, reg.predict(gmt_line[:, None]),
           color='#c0392b', lw=2,
           label=f'slope = {reg.coef_[0]:.2f} K/K\nr = {r:.2f}')
ax_sc.scatter(gmt[t_obs_f], tas_e[t_obs_f],
              s=80, color='black', zorder=5, label='Event')
ax_sc.set_xlabel('GMT anomaly (K)')
ax_sc.set_ylabel('Tas anomaly (K)')
ax_sc.legend(fontsize=9)
ax_sc.grid(True, alpha=0.2)
ax_sc.set_title('Tas vs GMT scatter', fontweight='bold')

plt.suptitle('GMT ↔ Temperature relationships', fontweight='bold', y=1.02)
plt.savefig(os.path.join(FIGURES_DIR, 'gmt_relationships.png'), dpi=200, bbox_inches='tight')
plt.show()

## Helper functions (shared by sections 3 & 4)

In [ ]:
def get_window(d, e_idx, is_factual):
    idx_arr = d['idx_f'] if is_factual else d['idx_c']
    t_obs   = idx_arr[e_idx]
    t0 = max(0, t_obs - WINDOW_BEFORE)
    t1 = min(d['f_tas'].shape[0], t_obs + WINDOW_AFTER)
    return t_obs, t0, t1


def plot_distributions(axes, d, e_idx, is_factual, cf_dict, title_prefix):
    """
    One subplot per method showing factual vs counterfactual Gaussian fits.
    axes : list of Axes, one per method.
    cf_dict : {method_name: counterfactual_array (T,)}
    """
    t_obs, t0, t1 = get_window(d, e_idx, is_factual)
    val     = d['f_tas_vals'][e_idx] if is_factual else d['c_tas_vals'][e_idx]
    tas_run = d['f_tas'][:, e_idx] if is_factual else d['c_tas'][:, e_idx]

    for ax, (name, cf) in zip(axes, cf_dict.items()):
        tas_win = tas_run[t0:t1]
        cf_win  = cf[t0:t1]
        xmin = min(tas_win.min(), cf_win.min()) - 0.3
        xmax = max(tas_win.max(), cf_win.max()) + 0.3
        x    = np.linspace(xmin, xmax, 300)

        mu_f,  s_f  = norm.fit(tas_win)
        mu_cf, s_cf = norm.fit(cf_win)
        p_f  = norm.sf(val, mu_f,  s_f)
        p_cf = norm.sf(val, mu_cf, s_cf)
        pn   = max(0.0, 1.0 - p_cf / p_f) if p_f > 0 else 0.0

        ax.fill_between(x, norm.pdf(x, mu_f,  s_f),  alpha=0.4,
                        color='#c0392b', label=f'Factual  μ={mu_f:.2f}')
        ax.fill_between(x, norm.pdf(x, mu_cf, s_cf), alpha=0.4,
                        color='#2980b9', label=f'CF  μ={mu_cf:.2f}')
        ax.axvline(val, color='black', lw=1.5, ls='--', label=f'u={val:.2f}')
        ax.set_title(f'{title_prefix}\n{name}\nPN={pn:.3f}',
                     fontsize=9, fontweight='bold')
        ax.legend(fontsize=7)
        ax.set_xlabel('Temperature anomaly (K)')
        ax.set_ylabel('Density')
        ax.grid(True, alpha=0.2)


def build_cf_dict(d, e_idx, is_factual):
    """
    Returns {method_name: counterfactual_series (T,)} for all methods,
    plus extras dict with fitted objects for later plotting.
    """
    tas_run = d['f_tas'][:, e_idx] if is_factual else d['c_tas'][:, e_idx]
    slp_run = d['f_slp'] if is_factual else d['c_slp']
    ev_lat, ev_lon = d['location'][e_idx]
    gmt_f   = d['gmt4_f']
    gmt_c   = np.full_like(gmt_f, gmt_f[0])

    # Ensemble ground truth
    t_obs, t0, t1 = get_window(d, e_idx, is_factual)
    pool_f = np.concatenate([
        m['f_tas'][t0:t1, e_idx] for m in full_data
        if e_idx < m['f_tas'].shape[1]])
    pool_c = np.concatenate([
        m['c_tas'][t0:t1, e_idx] for m in full_data
        if e_idx < m['c_tas'].shape[1]])
    # Pad to (T,) so indexing is consistent (we pass full arrays, slice inside plot_distributions)
    # For ensemble we directly build a padded stand-in — handled via special key below

    # Thermodynamic
    reg      = LinearRegression().fit(gmt_f[:, None], tas_run)
    cf_thermo = tas_run - reg.predict(gmt_f[:, None]) + reg.predict(gmt_c[:, None])

    # Local Ridge — no PCA, direct fit on the flattened box
    slp_local = extract_local_slp(
        slp_run, d['slp_lat'], d['slp_lon'], ev_lat, ev_lon, DYNADJ_HALF_DEG)
    ridge_l  = RidgeCV(alphas=RIDGE_ALPHAS).fit(slp_local, tas_run)
    cf_local = ridge_l.predict(slp_local)

    cf_dict = {
        'Thermo ML':   cf_thermo,
        'DynAdj Local': cf_local,
    }
    extras = {
        'ridge_l':  ridge_l,
        'slp_local': slp_local,
        'pool_f': pool_f, 'pool_c': pool_c,
        'tas_run': tas_run,
    }
    return cf_dict, extras


def _get_analogue_data(d, e_idx, is_factual):
    """
    Run KNN inside the local box and return per-analogue SLP arrays.
    Returns: anom_each (N,nlb,nlo_b), mean_each (N,nlb,nlo_b),
             lat_box, lon_box, ev_lat, ev_lon
    """
    slp_run  = d['f_slp'] if is_factual else d['c_slp']
    t_obs, _, _ = get_window(d, e_idx, is_factual)
    ev_lat, ev_lon = d['location'][e_idx]
    slp_lat, slp_lon = d['slp_lat'], d['slp_lon']
    hw       = ANALOGUE_HALF_DEG
    pool_end = N_YEARS_POOL * 12

    slp_local = extract_local_slp(
        slp_run, slp_lat, slp_lon, ev_lat, ev_lon, hw)

    pca_a    = PCA(n_components=min(20, slp_local.shape[1])).fit(slp_local)
    slp_apcs = pca_a.transform(slp_local)
    knn      = NearestNeighbors(n_neighbors=N_ANALOGUES).fit(slp_apcs[:pool_end])
    _, idx   = knn.kneighbors(slp_apcs[t_obs].reshape(1, -1))
    ana_idx  = idx[0]

    lat_idx  = np.where((slp_lat >= ev_lat - hw) & (slp_lat <= ev_lat + hw))[0]
    lon_idx  = np.where((slp_lon >= ev_lon - hw) & (slp_lon <= ev_lon + hw))[0]
    nlb, nlo_b = len(lat_idx), len(lon_idx)
    lat_box  = slp_lat[lat_idx]
    lon_box  = slp_lon[lon_idx]

    clim      = slp_local[:pool_end].mean(axis=0)
    anom_each = (slp_local[ana_idx] - clim).reshape(N_ANALOGUES, nlb, nlo_b)
    mean_each =  slp_local[ana_idx].reshape(N_ANALOGUES, nlb, nlo_b)
    return anom_each, mean_each, lat_box, lon_box, ev_lat, ev_lon


def plot_analogue_maps(d, e_idx, suptitle, save_path):
    """
    Big grid: one panel per analogue for both factual (top) and
    counterfactual (bottom) runs.

    Each panel:
      pcolormesh  — SLP anomaly vs box climatology (RdBu_r, shared scale)
      contours    — raw analogue SLP (thin black lines)
      star        — event barycentre
      Cartopy features zoomed to the box
    """
    import math
    from matplotlib.gridspec import GridSpec

    ncols   = math.ceil(math.sqrt(N_ANALOGUES))
    nrows_s = math.ceil(N_ANALOGUES / ncols)

    data_f  = _get_analogue_data(d, e_idx, is_factual=True)
    data_cf = _get_analogue_data(d, e_idx, is_factual=False)
    vmax = max(
        np.nanpercentile(np.abs(data_f[0]),  98),
        np.nanpercentile(np.abs(data_cf[0]), 98),
    )

    panel_w, panel_h = 2.4, 2.2
    fig = plt.figure(figsize=(ncols * panel_w,
                               nrows_s * 2 * panel_h + 1.5))

    gs_top = GridSpec(nrows_s, ncols, figure=fig,
                      top=0.91, bottom=0.51, hspace=0.05, wspace=0.05)
    gs_bot = GridSpec(nrows_s, ncols, figure=fig,
                      top=0.47, bottom=0.07, hspace=0.05, wspace=0.05)

    for scen_label, scenario_data, gs in [
        ('Factual analogues',        data_f,  gs_top),
        ('Counterfactual analogues', data_cf, gs_bot),
    ]:
        anom_each, mean_each, lat_box, lon_box, ev_lat, ev_lon = scenario_data
        for k in range(N_ANALOGUES):
            row = k // ncols
            col = k % ncols
            ax  = fig.add_subplot(gs[row, col], projection=PROJ)
            ax.set_extent([lon_box.min(), lon_box.max(),
                            lat_box.min(), lat_box.max()], crs=PROJ)
            add_map_features(ax)
            ax.pcolormesh(lon_box, lat_box, anom_each[k],
                           cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                           transform=PROJ, zorder=3)
            lon2d, lat2d = np.meshgrid(lon_box, lat_box)
            try:
                ax.contour(lon2d, lat2d, mean_each[k],
                            levels=6, colors='black', linewidths=0.4,
                            transform=PROJ, zorder=4, alpha=0.55)
            except Exception:
                pass
            ax.scatter([ev_lon], [ev_lat], s=30, color='black',
                        marker='*', transform=PROJ, zorder=6)
            if col == 0 and k == 0:
                ax.set_title(scen_label, fontsize=7, fontweight='bold', pad=2)
            else:
                ax.set_title(str(k + 1), fontsize=6, pad=1)

    sm = plt.cm.ScalarMappable(
        cmap='RdBu_r', norm=plt.Normalize(vmin=-vmax, vmax=vmax))
    sm.set_array([])
    cax = fig.add_axes([0.15, 0.02, 0.70, 0.018])
    fig.colorbar(sm, cax=cax, orientation='horizontal').set_label(
        f'SLP anomaly vs climatology (Pa)  —  box ±{ANALOGUE_HALF_DEG}°  '
        f'| {N_ANALOGUES} nearest analogues', fontsize=8)

    fig.suptitle(suptitle, fontweight='bold', fontsize=11)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

def plot_local_ridge_weights(d, e_idx, extras, suptitle, save_path):
    """
    Ridge weights (no PCA) for the DynAdj local box, zoomed to that box
    with full country / ocean / sea boundaries visible on top.
    Weights are the raw Ridge coefficients reshaped onto the lat/lon sub-grid.
    """
    ev_lat, ev_lon   = d['location'][e_idx]
    slp_lat, slp_lon = d['slp_lat'], d['slp_lon']
    hw               = DYNADJ_HALF_DEG

    # Direct Ridge coefficients — one weight per grid point in the box
    w_flat = extras['ridge_l'].coef_   # (n_pts_local,)

    lat_idx = np.where((slp_lat >= ev_lat - hw) & (slp_lat <= ev_lat + hw))[0]
    lon_idx = np.where((slp_lon >= ev_lon - hw) & (slp_lon <= ev_lon + hw))[0]
    nlb, nlo_b = len(lat_idx), len(lon_idx)
    lat_box = slp_lat[lat_idx]
    lon_box = slp_lon[lon_idx]

    w_2d = np.full((nlb, nlo_b), np.nan)
    if w_flat.shape[0] == nlb * nlo_b:
        w_2d = w_flat.reshape(nlb, nlo_b)

    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': PROJ})
    ax.set_extent([lon_box.min(), lon_box.max(),
                   lat_box.min(), lat_box.max()], crs=PROJ)

    # Draw weights first (zorder=1), then boundaries on top
    vmax = np.nanpercentile(np.abs(w_2d[~np.isnan(w_2d)]), 98)
    pcm  = ax.pcolormesh(lon_box, lat_box, w_2d, cmap='RdBu_r',
                          vmin=-vmax, vmax=vmax, transform=PROJ, zorder=1)

    # Land semi-transparent wash + coast + borders over the colour field
    ax.add_feature(LAND,    zorder=2, alpha=0.15)
    ax.add_feature(COAST,   zorder=3)
    ax.add_feature(BORDERS, zorder=3)

    ax.scatter([ev_lon], [ev_lat], s=150, color='black',
               marker='*', transform=PROJ, zorder=4, label='Event barycentre')
    ax.legend(loc='lower left', fontsize=8)

    plt.colorbar(pcm, ax=ax,
                 label=f'Ridge coefficient  (box ±{hw}°, direct — no PCA)',
                 shrink=0.85, pad=0.02)
    ax.set_title(suptitle, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()


## 3. Correctly attributed event (factual run)

Best-attributed factual event: distributions, analogue SLP maps, local Ridge weights.

In [ ]:
e_idx = e_fact
ev_lat, ev_lon = d['location'][e_idx]
print(f'Event {e_idx}: lat={ev_lat:.1f}  lon={ev_lon:.1f}  PN={pn_f[e_idx]:.3f}')

cf_dict, extras = build_cf_dict(d, e_idx, is_factual=True)

In [ ]:
# ── Distributions ─────────────────────────────────────────────────
# Add ensemble ground truth as first panel
t_obs, t0, t1 = get_window(d, e_idx, is_factual=True)
pool_f = extras['pool_f']
pool_c = extras['pool_c']
val    = d['f_tas_vals'][e_idx]

# Build cf_dict including ensemble (special handling: both factual and CF are pools)
all_methods = {'Ensemble (ground truth)': None, **cf_dict}
fig, axes = plt.subplots(1, len(all_methods), figsize=(5 * len(all_methods), 4))
if len(all_methods) == 1: axes = [axes]

# Ensemble panel
ax0 = axes[0]
x = np.linspace(min(pool_f.min(), pool_c.min()) - 0.3,
                max(pool_f.max(), pool_c.max()) + 0.3, 300)
mu_f, s_f   = norm.fit(pool_f)
mu_cf, s_cf = norm.fit(pool_c)
p_f  = norm.sf(val, mu_f,  s_f)
p_cf = norm.sf(val, mu_cf, s_cf)
pn_ens = max(0.0, 1.0 - p_cf / p_f) if p_f > 0 else 0.0
ax0.fill_between(x, norm.pdf(x, mu_f,  s_f),  alpha=0.4, color='#c0392b', label=f'Factual μ={mu_f:.2f}')
ax0.fill_between(x, norm.pdf(x, mu_cf, s_cf), alpha=0.4, color='#2980b9', label=f'CF μ={mu_cf:.2f}')
ax0.axvline(val, color='black', lw=1.5, ls='--', label=f'u={val:.2f}')
ax0.set_title(f'Ensemble (ground truth)\nPN={pn_ens:.3f}', fontsize=9, fontweight='bold')
ax0.legend(fontsize=7); ax0.grid(True, alpha=0.2)
ax0.set_xlabel('Temperature anomaly (K)'); ax0.set_ylabel('Density')

# Method panels
plot_distributions(axes[1:], d, e_idx, True, cf_dict, 'Correct attribution')

fig.suptitle(f'Correctly attributed event {e_idx}  —  Gaussian distributions',
             fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'correct_distributions.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── Analogue SLP maps ──────────────────────────────────────────────
plot_analogue_maps(
    d, e_idx,
    suptitle=f'Correct attribution — event {e_idx}  (lat={ev_lat:.1f}  lon={ev_lon:.1f})',
    save_path=os.path.join(FIGURES_DIR, 'correct_analogue_slp.png'))

In [ ]:
# ── Local Ridge weight map ─────────────────────────────────────────
plot_local_ridge_weights(
    d, e_idx, extras,
    suptitle=f'Correct attribution — Local Ridge weights  (event {e_idx}, box ±{LOCAL_HALF_DEG}°)',
    save_path=os.path.join(FIGURES_DIR, 'correct_ridge_weights.png'))

## 4. Type I error — spuriously attributed counterfactual event

Counterfactual event with the highest spurious PN: same three diagnostics.

In [ ]:
e_idx_cf = e_type1
ev_lat_cf, ev_lon_cf = d['location'][e_idx_cf]
print(f'Event {e_idx_cf}: lat={ev_lat_cf:.1f}  lon={ev_lon_cf:.1f}  spurious PN={pn_cf[e_idx_cf]:.3f}')

cf_dict_cf, extras_cf = build_cf_dict(d, e_idx_cf, is_factual=False)

In [ ]:
# ── Distributions ─────────────────────────────────────────────────
t_obs_cf, t0c, t1c = get_window(d, e_idx_cf, is_factual=False)
pool_f_cf = extras_cf['pool_f']
pool_c_cf = extras_cf['pool_c']
val_cf    = d['c_tas_vals'][e_idx_cf]

all_methods_cf = {'Ensemble (truth: PN≈0)': None, **cf_dict_cf}
fig2, axes2 = plt.subplots(1, len(all_methods_cf), figsize=(5 * len(all_methods_cf), 4))
if len(all_methods_cf) == 1: axes2 = [axes2]

# Ensemble panel
ax0 = axes2[0]
x2 = np.linspace(min(pool_f_cf.min(), pool_c_cf.min()) - 0.3,
                 max(pool_f_cf.max(), pool_c_cf.max()) + 0.3, 300)
mu_f2, s_f2   = norm.fit(pool_f_cf)
mu_cf2, s_cf2 = norm.fit(pool_c_cf)
p_f2   = norm.sf(val_cf, mu_f2,  s_f2)
p_cf2  = norm.sf(val_cf, mu_cf2, s_cf2)
pn_ens2 = max(0.0, 1.0 - p_cf2 / p_f2) if p_f2 > 0 else 0.0
ax0.fill_between(x2, norm.pdf(x2, mu_f2,  s_f2),  alpha=0.4, color='#c0392b', label=f'Factual μ={mu_f2:.2f}')
ax0.fill_between(x2, norm.pdf(x2, mu_cf2, s_cf2), alpha=0.4, color='#2980b9', label=f'CF μ={mu_cf2:.2f}')
ax0.axvline(val_cf, color='black', lw=1.5, ls='--', label=f'u={val_cf:.2f}')
ax0.set_title(f'Ensemble (truth: PN≈0)\nPN={pn_ens2:.3f}', fontsize=9, fontweight='bold')
ax0.legend(fontsize=7); ax0.grid(True, alpha=0.2)
ax0.set_xlabel('Temperature anomaly (K)'); ax0.set_ylabel('Density')

plot_distributions(axes2[1:], d, e_idx_cf, False, cf_dict_cf, 'Type I error')

fig2.suptitle(f'Type I error — counterfactual event {e_idx_cf}  (spurious attribution)',
              fontweight='bold', color='#c0392b')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'type1_distributions.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── Analogue SLP maps ──────────────────────────────────────────────
plot_analogue_maps(
    d, e_idx_cf,
    suptitle=f'Type I error — event {e_idx_cf}  (lat={ev_lat_cf:.1f}  lon={ev_lon_cf:.1f})',
    save_path=os.path.join(FIGURES_DIR, 'type1_analogue_slp.png'))

In [ ]:
# ── Local Ridge weight map ─────────────────────────────────────────
plot_local_ridge_weights(
    d, e_idx_cf, extras_cf,
    suptitle=f'Type I error — Local Ridge weights  (event {e_idx_cf}, box ±{LOCAL_HALF_DEG}°)',
    save_path=os.path.join(FIGURES_DIR, 'type1_ridge_weights.png'))

## Summary of saved figures

| File | Content |
|---|---|
| `freq_map.png` | Event frequency map + barycentres |
| `gmt_relationships.png` | GMT ↔ temperature for the best-attributed event |
| `correct_distributions.png` | Factual vs CF distributions — best attributed event |
| `correct_analogue_slp.png` | Analogue SLP mean + anomaly — best attributed event |
| `correct_ridge_weights.png` | Local Ridge weights — best attributed event |
| `type1_distributions.png` | Factual vs CF distributions — Type I error event |
| `type1_analogue_slp.png` | Analogue SLP mean + anomaly — Type I error event |
| `type1_ridge_weights.png` | Local Ridge weights — Type I error event |